# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [25]:
import pandas as pd
import numpy as np
from IPython.display import display

# Load the three datasets
url1 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
url2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
url3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"

df1 = pd.read_csv(url1)
df2 = pd.read_csv(url2)
df3 = pd.read_csv(url3)

print("Original shapes")
print("File 1:", df1.shape, "| File 2:", df2.shape, "| File 3:", df3.shape)

# Clean and standardize column names before combining
def clean_column_names(dataframe):
    dataframe = dataframe.copy()
    dataframe.columns = dataframe.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
    dataframe = dataframe.rename(columns={"state": "st"})
    return dataframe

df1 = clean_column_names(df1)
df2 = clean_column_names(df2)
df3 = clean_column_names(df3)

print("\nFile 1 columns:", df1.columns.tolist())
print("File 2 columns:", df2.columns.tolist())
print("File 3 columns:", df3.columns.tolist())

# Combine the three datasets vertically
df = pd.concat([df1, df2, df3], axis=0, ignore_index=True)

print("\nCombined shape before cleaning:", df.shape)

# Remove rows where every column is empty
df = df.dropna(how="all").reset_index(drop=True)

# Clean text spacing
text_columns = df.select_dtypes(include=["object", "string"]).columns
for column in text_columns:
    df[column] = df[column].astype("string").str.strip()

# Standardize gender values
gender_mapping = {"M": "Male", "male": "Male", "Male": "Male", "F": "Female", "female": "Female", "Femal": "Female", "Female": "Female"}
df["gender"] = df["gender"].replace(gender_mapping)

# Standardize state values
state_mapping = {"AZ": "Arizona", "Cali": "California", "WA": "Washington"}
df["st"] = df["st"].replace(state_mapping)

# Standardize education values
df["education"] = df["education"].replace({"Bachelors": "Bachelor"})

# Standardize vehicle class values
df["vehicle_class"] = df["vehicle_class"].replace({"Sports Car": "Luxury", "Luxury SUV": "Luxury", "Luxury Car": "Luxury"})

# Clean customer lifetime value
df["customer_lifetime_value"] = df["customer_lifetime_value"].astype("string").str.replace("%", "", regex=False).str.replace(",", "", regex=False).str.strip()
df["customer_lifetime_value"] = pd.to_numeric(df["customer_lifetime_value"], errors="coerce")

# Clean number of open complaints
def clean_complaints(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    if "/" in value:
        parts = value.split("/")
        return pd.to_numeric(parts[1], errors="coerce") if len(parts) > 1 else np.nan
    return pd.to_numeric(value, errors="coerce")

df["number_of_open_complaints"] = df["number_of_open_complaints"].apply(clean_complaints)

# Convert numerical columns
numeric_columns = ["income", "monthly_premium_auto", "total_claim_amount"]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Check missing values before filling
print("\nMissing values before filling")
display(df.isnull().sum().sort_values(ascending=False).to_frame("missing_values"))

# Fill missing categorical values using the mode
categorical_columns = df.select_dtypes(include=["object", "string"]).columns
for column in categorical_columns:
    mode_value = df[column].mode(dropna=True)
    if not mode_value.empty:
        df[column] = df[column].fillna(mode_value.iloc[0])

# Fill missing numerical values using the median
numeric_columns = df.select_dtypes(include="number").columns
for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

# Remove exact duplicate rows
duplicates_before = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

# Round numerical values
round_columns = ["customer_lifetime_value", "monthly_premium_auto", "total_claim_amount"]
df[round_columns] = df[round_columns].round(2)

# Convert suitable columns to integers
integer_columns = ["income", "number_of_open_complaints"]
for column in integer_columns:
    df[column] = df[column].round().astype(int)

# Arrange columns in a clean order
column_order = ["customer", "st", "gender", "education", "customer_lifetime_value", "income", "monthly_premium_auto", "number_of_open_complaints", "policy_type", "vehicle_class", "total_claim_amount"]
df = df[[column for column in column_order if column in df.columns]]

# Final validation
print("\nCleaning results")
print("Final shape:", df.shape)
print("Duplicates removed:", duplicates_before)
print("Remaining duplicate rows:", df.duplicated().sum())
print("Remaining missing values:", df.isnull().sum().sum())

print("\nFinal data types")
display(df.dtypes.to_frame("data_type"))

print("\nFirst 10 cleaned rows")
display(df.head(10))

print("\nNumerical summary")
display(df.describe().round(2))

print("\nCategorical summary")
display(df.describe(include=["object", "string"]))

# Save the cleaned combined dataset
df.to_csv("cleaned_customer_data.csv", index=False)

print("\nThe cleaned dataset was saved as cleaned_customer_data.csv")

Original shapes
File 1: (4008, 11) | File 2: (996, 11) | File 3: (7070, 11)

File 1 columns: ['customer', 'st', 'gender', 'education', 'customer_lifetime_value', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'policy_type', 'vehicle_class', 'total_claim_amount']
File 2 columns: ['customer', 'st', 'gender', 'education', 'customer_lifetime_value', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'total_claim_amount', 'policy_type', 'vehicle_class']
File 3 columns: ['customer', 'st', 'customer_lifetime_value', 'education', 'gender', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'policy_type', 'total_claim_amount', 'vehicle_class']

Combined shape before cleaning: (12074, 11)

Missing values before filling


,missing_values
gender,122
customer_lifetime_value,7
customer,0
st,0
education,0
income,0
monthly_premium_auto,0
number_of_open_complaints,0
policy_type,0
vehicle_class,0



Cleaning results
Final shape: (9134, 11)
Duplicates removed: 3
Remaining duplicate rows: 0
Remaining missing values: 0

Final data types


,data_type
customer,string
st,string
gender,string
education,string
customer_lifetime_value,Float64
income,int64
monthly_premium_auto,float64
number_of_open_complaints,int64
policy_type,string
vehicle_class,string



First 10 cleaned rows


...


Numerical summary


,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,total_claim_amount
count,9134.0,9134.00,9134.00,9134.00,9134.00
mean,181937.9,37824.85,110.39,0.38,430.48
std,440903.85,30359.23,581.47,0.91,289.62
min,1898.01,0.00,61.00,0.00,0.10
25%,4650.06,0.00,68.00,0.00,266.96
50%,7714.88,34240.00,83.00,0.00,377.50
75%,26131.72,62446.50,109.00,0.00,546.05
max,5816655.35,99981.00,35354.00,5.00,2893.24



Categorical summary


,customer,st,gender,education,policy_type,vehicle_class
count,9134,9134,9134,9134,9134,9134
unique,9056,5,2,5,3,4
top,GA49547,California,Female,Bachelor,Personal Auto,Four-Door Car
freq,2,3150,4726,2742,6790,4640



The cleaned dataset was saved as cleaned_customer_data.csv


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [26]:
# Your code goes here
import pandas as pd
import numpy as np
from IPython.display import display

In [27]:
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"

df = pd.read_csv(url)

In [28]:
sales_channel = pd.pivot_table(
    df,
    index="sales_channel",
    values="total_claim_amount",
    aggfunc="sum"
).round(2)

sales_channel.columns = ["Total Revenue"]

display(sales_channel)

,Total Revenue
sales_channel,
Agent,1810226.82
Branch,1301204.00
Call Center,926600.82
Web,706600.04


In [29]:
values="total_claim_amount"

In [30]:
values="monthly_premium_auto"

In [31]:
clv_pivot = pd.pivot_table(
    df,
    index="education",
    columns="gender",
    values="customer_lifetime_value",
    aggfunc="mean"
).round(2)

display(clv_pivot)

gender,F,M
education,,
Bachelor,7874.27,7703.60
College,7748.82,8052.46
Doctor,7328.51,7415.33
High School or Below,8675.22,8149.69
Master,8157.05,8168.83


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [32]:
# Your code goes here
import pandas as pd
from IPython.display import display

In [33]:
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"

df = pd.read_csv(url)

In [34]:
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"])

In [35]:
df["month"] = df["effective_to_date"].dt.month_name()

In [36]:
complaints_summary = (
    df.groupby(["policy_type", "month"])["number_of_open_complaints"]
      .count()
      .reset_index(name="Number of Complaints")
)

display(complaints_summary)

,policy_type,month,Number of Complaints
0,Corporate Auto,February,1089
1,Corporate Auto,January,1252
2,Personal Auto,February,3799
3,Personal Auto,January,4329
4,Special Auto,February,204
5,Special Auto,January,237


In [37]:
complaints_summary = complaints_summary.sort_values(
    by=["policy_type", "month"]
)

display(complaints_summary)

,policy_type,month,Number of Complaints
0,Corporate Auto,February,1089
1,Corporate Auto,January,1252
2,Personal Auto,February,3799
3,Personal Auto,January,4329
4,Special Auto,February,204
5,Special Auto,January,237


In [38]:
complaints_wide = pd.pivot_table(
    df,
    index="policy_type",
    columns="month",
    values="number_of_open_complaints",
    aggfunc="count",
    fill_value=0
)

display(complaints_wide)

month,February,January
policy_type,,
Corporate Auto,1089,1252
Personal Auto,3799,4329
Special Auto,204,237


In [39]:
complaints_long = (
    complaints_wide
    .reset_index()
    .melt(
        id_vars="policy_type",
        var_name="Month",
        value_name="Number of Complaints"
    )
)

display(complaints_long)

,policy_type,Month,Number of Complaints
0,Corporate Auto,February,1089
1,Personal Auto,February,3799
2,Special Auto,February,204
3,Corporate Auto,January,1252
4,Personal Auto,January,4329
5,Special Auto,January,237
